In [3]:
import torch
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
model_name="google/flan-t5-small"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)
def local_llm(prompt):
    if hasattr(prompt,"to_string"):
        prompt=prompt.to_string()
    inputs=tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )
    with torch.no_grad():
        output_ids=model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )
    answer=tokenizer.decode(output_ids[0],skip_special_tokens=True)
    return answer
@tool
def nutrition_tool(goal:str) -> str:
    """Give nutrition advice based on a fitness goal."""
    goal=goal.lower()
    if "fat" in goal or "loss" in goal or "cut" in goal:
        return "For fat loss,eat high protein,low fat, moderate carbs,and stay in a calorie deficit."

    if "muscle" in goal or "gain" in goal or "bulk" in goal:
       return "For muscle gain,eat enough protein,stay in calorie surplus,and train with progressive overload." 
        
    return "For general fitness,eat balance meals with protein,carbs,healthy fats, and enough vegetables."

@tool
def workout_tool(goal:str) -> str:
    """Give workout advice based on a fitness goal"""
    goal=goal.lower()
    if "chest" in goal:
        return "For chest training,do bench press,incline dumbbell press,and cable fly."
    
    if "iegs" in goal:
        return "For legs and glutes,do squats,iunges,hip trusts,and Romanian deadlifts."
    
    if "back" in goal:
        return "For back training,do pull_ups,rows,lat pulldowns,and deadlifts."
    
    return "For general training,combine compund lifts,accessory exercises, and progressive overload."
tools={
    "nutrition":nutrition_tool,
    "workout":workout_tool
}
def choose_tool(question):
    question_lower=question.lower()
    
    nutrition_keywards=[
        "eat","food","diet","nutrition","calorie","protein","fat loss","muscle gain","dinner"
    ]
    workout_keywards=[
        "exercise","workout","train","training","chest","legs","glutes","back"
    ]
    for keyward in nutrition_keywards:
        if keyward in question_lower:
            return "nutrition"
    for keyward in workout_keywards:
        if keyward in question_lower:
            return "workout"
    return "nutrition"
prompt_template=PromptTemplate.from_template(
    """
You are a helpful fitness assistant.
User qusetion:
{question}
Tool result:
{tool_result}
Use the tool result to answer the user clearly.
Answer:
"""
)
llm_runnable=RunnableLambda(local_llm)
output_parser=StrOutputParser()
chain=prompt_template|llm_runnable|output_parser

def fitness_agent(question):
    selected_tool_name=choose_tool(question)
    selected_tool=tools[selected_tool_name]

    tool_result=selected_tool.invoke({
        "goal":question
    })
    final_answer=chain.invoke({
        "question":question,
        "tool_result":tool_result
    })
    return {
        "question":question,
        "selected_tool":selected_tool_name,
        "tool_result":tool_result,
        "final_answer":final_answer
    }
test_questions=[
    "What should I eat during fat loss?",
    "Waht exercises should I do for chest day?",
    "How do I train legs and glutes?",
    "How should I eat during muscle gain?"
]
for question in test_questions:
    result=fitness_agent(question)

print("=============================")
print("Question",result["question"])
print("Selected_tool",result["selected_tool"])
print("Tool result",result["tool_result"])
print("Final answer",result["final_answer"])



Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4660.12it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question How should I eat during muscle gain?
Selected_tool nutrition
Tool result For muscle gain,eat enough protein,stay in calorie surplus,and train with progressive overload.
Final answer For muscle gain,eat enough protein,stay in calorie surplus,and train with progressive overload.


In [ ]:
["eat","diet","food","calorie","protein","fat loss","muscle gain"]
["train","workout","exercise","chest","legs","back","glutes"]
"For fat loss,focus on high protein,moderate carbs,low fat,and calorie comtrol.Your current goal is"
"For muscle gain,eat enough protein and stay in a calorie surplus.Your weight is"
"Eat balanced meals with protein ,carbs,healthy fats,and vegetables."
"For chest training,use bench press,incline dumbbell press,and cable fly."
"For legs and glutes,use squats,hip trusts,lunges,and Romanian deadlifts."
"Use progressive overload and choose training volume based on your level:{}"
"Fat loss requires a calories deficit and enough protein"
"Muscle gain requires calories surplus,progressive overload and enough recovery."
"Chest training can include bench press,incline dumbbell press, and cable fly."
"Leg training can include squats,lunges,hip trusts,and Romanian deadlifts"
